In [1]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import joblib
from catboost import Pool
from sklearn.metrics import balanced_accuracy_score

sys.path.insert(0, os.path.abspath('..'))

from src.models.predict import load_cat_models, load_lgb_models, wrap_lgb_model
from src.models.train import LABEL_MAP, REVERSE_MAP

warnings.filterwarnings('ignore')

project_root_directory = Path('..').resolve()
processed_data_directory = project_root_directory / 'data' / 'processed'
trained_models_directory = project_root_directory / 'models'
figures_report_directory = project_root_directory / 'reports' / 'figures'
outputs_directory = project_root_directory / 'outputs'
figures_report_directory.mkdir(parents=True, exist_ok=True)
outputs_directory.mkdir(parents=True, exist_ok=True)

training_features = pd.read_csv(processed_data_directory / 'X_train.csv')
training_labels = pd.read_csv(processed_data_directory / 'y_train.csv').squeeze()

print(f'Feature columns ({len(training_features.columns)}): {list(training_features.columns)}')


10:48:05 | INFO     | Logger ready — log file: ../logs\training_20260727_104805.log


Feature columns (53): ['alpha', 'delta', 'delta_abs', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z', 'g_i', 'r_z', 'u_r', 'g_z', 'u_i', 'u_z', 'r_g', 'g_div_z', 'r_div_i', 'u_div_r', 'g_div_r', 'gz_x_ri', 'gr_x_iz', 'band_mean', 'band_std', 'band_min', 'band_max', 'band_range', 'band_sum', 'band_skew', 'band_median', 'band_iqr', 'band_cv', 'redshift_log1p', 'redshift_sq', 'redshift_sqrt', 'redshift_abs', 'is_high_z', 'is_star_z', 'is_redshift_zero', 'is_very_high_z', 'is_negative_z', 'redshift_x_gz', 'redshift_x_bandstd', 'redshift_x_bandmean', 'redshift_over_bandmean', 'g_r_x_redshift', 'u_g_x_redshift', 'spectral_type_enc', 'galaxy_population_enc']


In [2]:
print('Loading production models trained in Notebook 03...')

lightgbm_models = load_lgb_models(str(trained_models_directory))
lightgbm_models = [
    wrap_lgb_model(model) if hasattr(model, 'predict') and not hasattr(model, 'predict_proba') else model
    for model in lightgbm_models
]
catboost_models = load_cat_models(str(trained_models_directory))

print(f'Loaded {len(lightgbm_models)} LightGBM fold models')
print(f'Loaded {len(catboost_models)} CatBoost fold models')
print('✗ XGBoost intentionally not loaded — blend weight is 0.0 in production')


Loading production models trained in Notebook 03...
Loaded 10 LightGBM from 'D:\Dev\predicting-stellar-class\models'
Loaded 10 CatBoost from 'D:\Dev\predicting-stellar-class\models'
Loaded 10 LightGBM fold models
Loaded 10 CatBoost fold models
✗ XGBoost intentionally not loaded — blend weight is 0.0 in production


In [ ]:
BLEND_WEIGHTS = {'lgb': 0.85, 'xgb': 0.0, 'cat': 0.15}

feature_names = training_features.columns.tolist()
class_map = REVERSE_MAP 

print('Blend weights:', BLEND_WEIGHTS)
print('Class map:', class_map)


Blend weights: {'lgb': 0.85, 'xgb': 0.0, 'cat': 0.15}
Class map: {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}


In [ ]:
def explain_with_shap_production(model, validation_features, output_directory,
                                   sample_display_count=200, top_feature_count=12,
                                   sample_instance_index=0):
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)

    sampled_features = validation_features.sample(
        n=min(sample_display_count, len(validation_features)), random_state=42
    )

    raw_shap_values = np.asarray(
        model.get_feature_importance(type='ShapValues', data=Pool(sampled_features))
    )

    if raw_shap_values.shape[2] == sampled_features.shape[1] + 1:
        shap_values = raw_shap_values[:, :, :-1]
        base_value_array = raw_shap_values[:, :, -1]
    else:
        raise ValueError(f'Unexpected CatBoost SHAP shape: {raw_shap_values.shape}')

    predicted_class_indices = np.argmax(model.predict_proba(sampled_features), axis=1)
    dominant_class_index = int(
        np.bincount(predicted_class_indices, minlength=shap_values.shape[1]).argmax()
    )

    target_shap_values = shap_values[:, dominant_class_index, :]
    target_base_values = base_value_array[:, dominant_class_index]

    shap_explanation = shap.Explanation(
        values=target_shap_values,
        base_values=target_base_values,
        data=sampled_features,
        feature_names=sampled_features.columns.tolist(),
    )

    plt.figure(figsize=(12, 7))
    shap.plots.beeswarm(shap_explanation, max_display=top_feature_count, show=False)
    plt.tight_layout()
    plt.savefig(output_directory / 'shap_beeswarm_production.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(10, 6))
    shap.plots.bar(shap_explanation, max_display=top_feature_count, show=False)
    plt.tight_layout()
    plt.savefig(output_directory / 'shap_feature_importance_production.png', dpi=300, bbox_inches='tight')
    plt.close()

    return shap_explanation, sampled_features


production_catboost_model = catboost_models[0]

shap_explanation, shap_sample = explain_with_shap_production(
    production_catboost_model,
    training_features,
    output_directory=figures_report_directory,
    sample_display_count=200,
    top_feature_count=12,
)

print('Saved shap_beeswarm_production.png and shap_feature_importance_production.png')
print(f'these replace the pseudo-labeled versions for report section 6.7')


Saved shap_beeswarm_production.png and shap_feature_importance_production.png
these replace the pseudo-labeled versions for report section 6.7


In [ ]:
production_bundle = {
    'lightgbm_models': lightgbm_models,
    'catboost_models': catboost_models,
    'blend_weights': BLEND_WEIGHTS,
    'feature_names': feature_names,
    'class_map': class_map,
    'shap_background_sample': training_features.sample(n=200, random_state=42).reset_index(drop=True),
}

bundle_path = trained_models_directory / 'production_bundle.pkl'
joblib.dump(production_bundle, bundle_path)
print(f'Saved production bundle to {bundle_path}')
print(f'File size: {bundle_path.stat().st_size / 1e6:.1f} MB')


Saved production bundle to D:\Dev\predicting-stellar-class\models\production_bundle.pkl
File size: 137.9 MB


In [6]:
reloaded_bundle = joblib.load(bundle_path)

def bundle_predict_proba(bundle, features):
    lgb_pred = np.zeros((len(features), 3))
    for m in bundle['lightgbm_models']:
        lgb_pred += m.predict_proba(features)
    lgb_pred /= len(bundle['lightgbm_models'])

    cat_pred = np.zeros((len(features), 3))
    for m in bundle['catboost_models']:
        cat_pred += m.predict_proba(features)
    cat_pred /= len(bundle['catboost_models'])

    w = bundle['blend_weights']
    return lgb_pred * w['lgb'] + cat_pred * w['cat']

check_probs = bundle_predict_proba(reloaded_bundle, training_features)
check_preds = check_probs.argmax(axis=1)
encoded_labels = training_labels.map(LABEL_MAP).astype(int)

resub_accuracy = balanced_accuracy_score(encoded_labels, check_preds)
print(f'Resubstitution balanced accuracy (bundle, in-sample — not OOF, sanity check only): {resub_accuracy:.5f}')
print('This number will look higher than the real 0.965 OOF score since it is scored on training')
print('rows the models were fit on — it only confirms the bundle loads and predicts, nothing more.')


Resubstitution balanced accuracy (bundle, in-sample — not OOF, sanity check only): 0.97665
This number will look higher than the real 0.965 OOF score since it is scored on training
rows the models were fit on — it only confirms the bundle loads and predicts, nothing more.
